# Demo 1b: CDC Debezium-mal és Kafkával

BME Adatmérnökség – 2. hét

Ebben a demóban:
1. Debezium PostgreSQL connector regisztrálása
2. INSERT/UPDATE/DELETE végrehajtása és CDC események
3. Kafka topic-ok vizsgálata
4. Python consumer: változások valós idejű feldolgozása

In [1]:
import requests
import json
import psycopg2
import psycopg2.extras
from confluent_kafka import Consumer, KafkaError
import time

# Debezium Connect REST API
CONNECT_URL = "http://debezium:8083"

# Ellenőrzés: Debezium Connect elérhető?
resp = requests.get(f"{CONNECT_URL}/")
print(f"Kafka Connect verzió: {resp.json()['version']}")
print(f"Commit: {resp.json()['commit']}")

Kafka Connect verzió: 3.6.1
Commit: 5e3c2b738d253ff5


## 1. Debezium PostgreSQL Connector regisztrálása

In [2]:
# Connector konfiguráció
connector_config = {
    "name": "webshop-connector",
    "config": {
        "connector.class": "io.debezium.connector.postgresql.PostgresConnector",
        "database.hostname": "postgres",
        "database.port": "5432",
        "database.user": "dataeng",
        "database.password": "dataeng2024",
        "database.dbname": "webshop",
        "topic.prefix": "webshop",
        "plugin.name": "pgoutput",
        "publication.name": "dbz_publication",
        "slot.name": "debezium_slot",
        "schema.include.list": "public",
        "table.include.list": "public.customers,public.orders,public.order_items,public.products",
        "decimal.handling.mode": "double",
        "time.precision.mode": "connect"
    }
}

# Először töröljük ha már létezik
requests.delete(f"{CONNECT_URL}/connectors/webshop-connector")
time.sleep(2)

# Regisztráljuk a connectort
resp = requests.post(
    f"{CONNECT_URL}/connectors",
    headers={"Content-Type": "application/json"},
    json=connector_config
)
print(f"Státusz: {resp.status_code}")
print(json.dumps(resp.json(), indent=2))

Státusz: 201
{
  "name": "webshop-connector",
  "config": {
    "connector.class": "io.debezium.connector.postgresql.PostgresConnector",
    "database.hostname": "postgres",
    "database.port": "5432",
    "database.user": "dataeng",
    "database.password": "dataeng2024",
    "database.dbname": "webshop",
    "topic.prefix": "webshop",
    "plugin.name": "pgoutput",
    "publication.name": "dbz_publication",
    "slot.name": "debezium_slot",
    "schema.include.list": "public",
    "table.include.list": "public.customers,public.orders,public.order_items,public.products",
    "decimal.handling.mode": "double",
    "time.precision.mode": "connect",
    "name": "webshop-connector"
  },
  "tasks": [],
  "type": "source"
}


In [3]:
# Connector státusz ellenőrzés
time.sleep(5)  # Várunk a connector indulására

resp = requests.get(f"{CONNECT_URL}/connectors/webshop-connector/status")
status = resp.json()

print(f"Connector: {status['name']}")
print(f"  Típus: {status['type']}")
print(f"  Státusz: {status['connector']['state']}")
for task in status.get('tasks', []):
    print(f"  Task {task['id']}: {task['state']}")

# Kafka topic-ok listázása
print("\nDebezium által létrehozott topic-ok:")
resp = requests.get(f"{CONNECT_URL}/connectors/webshop-connector/topics")
topics = resp.json()
for connector, topic_list in topics.items():
    for t in topic_list.get('topics', []):
        print(f"  → {t}")

Connector: webshop-connector
  Típus: source
  Státusz: RUNNING
  Task 0: RUNNING

Debezium által létrehozott topic-ok:
  → webshop.public.customers
  → webshop.public.products
  → webshop.public.orders
  → webshop.public.order_items


## 2. Változások végrehajtása PostgreSQL-ben

Most végrehajtunk INSERT, UPDATE és DELETE műveleteket, majd megfigyeljük a CDC eseményeket a Kafka topic-okban.

In [4]:
# PostgreSQL kapcsolat
conn = psycopg2.connect(
    host="postgres", port=5432,
    dbname="webshop", user="dataeng", password="dataeng2024"
)
conn.autocommit = True
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

print("=== Változások végrehajtása ===\n")

# 1. INSERT - új ügyfél
cur.execute("""
    INSERT INTO customers (name, email, city, segment)
    VALUES ('Demo Béla', 'demo.bela@example.com', 'Debrecen', 'gold')
    RETURNING customer_id
""")
new_id = cur.fetchone()['customer_id']
print(f"[INSERT] Új ügyfél: #{new_id} – Demo Béla")

time.sleep(1)

# 2. UPDATE - szegmens módosítás
cur.execute(f"""
    UPDATE customers SET segment = 'premium', city = 'Budapest' 
    WHERE customer_id = {new_id}
""")
print(f"[UPDATE] Ügyfél #{new_id}: segment → premium, city → Budapest")

time.sleep(1)

# 3. INSERT - új rendelés
cur.execute(f"""
    INSERT INTO orders (customer_id, order_date, status, total_amount, shipping_city)
    VALUES ({new_id}, now(), 'pending', 149990, 'Budapest')
    RETURNING order_id
""")
order_id = cur.fetchone()['order_id']
print(f"[INSERT] Új rendelés: #{order_id}")

time.sleep(1)

# 4. UPDATE - rendelés státusz
cur.execute(f"UPDATE orders SET status = 'confirmed' WHERE order_id = {order_id}")
print(f"[UPDATE] Rendelés #{order_id}: status → confirmed")

time.sleep(1)

# 5. DELETE
cur.execute(f"DELETE FROM orders WHERE order_id = {order_id}")
print(f"[DELETE] Rendelés #{order_id} törölve")

print("\nVáltózások végrehajtva! Várunk 3 mp-et a Debezium propagálásra...")
time.sleep(3)

=== Változások végrehajtása ===

[INSERT] Új ügyfél: #37 – Demo Béla
[UPDATE] Ügyfél #37: segment → premium, city → Budapest
[INSERT] Új rendelés: #103
[UPDATE] Rendelés #103: status → confirmed
[DELETE] Rendelés #103 törölve

Váltózások végrehajtva! Várunk 3 mp-et a Debezium propagálásra...


## 3. CDC események olvasása Kafkából

A Debezium minden változást Kafka topic-okba ír. A topic neve: `webshop.public.<táblanév>`

In [5]:
# Kafka Consumer - CDC események olvasása
consumer = Consumer({
    'bootstrap.servers': 'kafka:9092',
    'group.id': 'demo1b-consumer',
    'auto.offset.reset': 'earliest',
    'enable.auto.commit': True,
})

# Feliratkozás a customers és orders topic-okra
topics = ['webshop.public.customers', 'webshop.public.orders']
consumer.subscribe(topics)
print(f"Feliratkozva: {topics}\n")

print("=== CDC események ===\n")
msg_count = 0
max_messages = 20
empty_polls = 0

while msg_count < max_messages and empty_polls < 10:
    msg = consumer.poll(2.0)
    
    if msg is None:
        empty_polls += 1
        continue
    
    if msg.error():
        if msg.error().code() == KafkaError._PARTITION_EOF:
            continue
        print(f"Hiba: {msg.error()}")
        break
    
    empty_polls = 0
    msg_count += 1
    
    value = json.loads(msg.value().decode('utf-8'))
    payload = value.get('payload', value)
    
    op_map = {'c': 'INSERT', 'u': 'UPDATE', 'd': 'DELETE', 'r': 'READ'}
    op = op_map.get(payload.get('op', '?'), '?')
    
    print(f"[{msg_count}] Topic: {msg.topic()} | Művelet: {op}")
    
    if payload.get('before'):
        print(f"     BEFORE: {json.dumps(payload['before'], ensure_ascii=False)[:120]}")
    if payload.get('after'):
        print(f"     AFTER:  {json.dumps(payload['after'], ensure_ascii=False)[:120]}")
    print()

consumer.close()
print(f"\nÖsszesen {msg_count} CDC esemény feldolgozva.")

Feliratkozva: ['webshop.public.customers', 'webshop.public.orders']

=== CDC események ===

[1] Topic: webshop.public.orders | Művelet: READ
     AFTER:  {"order_id": 1, "customer_id": 2, "order_date": 1766402076878, "status": "delivered", "total_amount": 594960.0, "shippin

[2] Topic: webshop.public.orders | Művelet: READ
     AFTER:  {"order_id": 2, "customer_id": 3, "order_date": 1771586076878, "status": "delivered", "total_amount": 259980.0, "shippin

[3] Topic: webshop.public.orders | Művelet: READ
     AFTER:  {"order_id": 3, "customer_id": 4, "order_date": 1770232476878, "status": "confirmed", "total_amount": 84980.0, "shipping

[4] Topic: webshop.public.orders | Művelet: READ
     AFTER:  {"order_id": 4, "customer_id": 5, "order_date": 1766517276878, "status": "cancelled", "total_amount": 59980.0, "shipping

[5] Topic: webshop.public.orders | Művelet: READ
     AFTER:  {"order_id": 5, "customer_id": 6, "order_date": 1769530476878, "status": "pending", "total_amount": 1149950.0,

## 4. CDC esemény struktúra részletesen

Vizsgáljuk meg egy Debezium CDC esemény teljes struktúráját.

In [6]:
# Egy esemény részletes vizsgálata
consumer2 = Consumer({
    'bootstrap.servers': 'kafka:9092',
    'group.id': 'demo1b-detail',
    'auto.offset.reset': 'earliest',
    'enable.auto.commit': True,
})
consumer2.subscribe(['webshop.public.customers'])

print("=== Debezium CDC esemény struktúra ===\n")

msg = None
for _ in range(30):
    msg = consumer2.poll(2.0)
    if msg and not msg.error():
        break

if msg and not msg.error():
    value = json.loads(msg.value().decode('utf-8'))
    
    # Schema info
    if 'schema' in value:
        print("Schema mezők:")
        print(f"  type: {value['schema'].get('type')}")
        print(f"  name: {value['schema'].get('name')}")
    
    payload = value.get('payload', value)
    print(f"\nPayload mezők: {list(payload.keys())}")
    print(f"\n  op: {payload.get('op')} (c=create, u=update, d=delete, r=read)")
    print(f"  ts_ms: {payload.get('ts_ms')}")
    
    if payload.get('source'):
        src = payload['source']
        print(f"\n  source.connector: {src.get('connector')}")
        print(f"  source.db: {src.get('db')}")
        print(f"  source.schema: {src.get('schema')}")
        print(f"  source.table: {src.get('table')}")
        print(f"  source.lsn: {src.get('lsn')}")
    
    print(f"\n  before: {json.dumps(payload.get('before'), ensure_ascii=False, indent=4)[:200] if payload.get('before') else 'null'}")
    print(f"\n  after: {json.dumps(payload.get('after'), ensure_ascii=False, indent=4)[:200] if payload.get('after') else 'null'}")
else:
    print("Nem sikerült eseményt olvasni. Próbáld újra a cellát!")

consumer2.close()

=== Debezium CDC esemény struktúra ===

Schema mezők:
  type: struct
  name: webshop.public.customers.Envelope

Payload mezők: ['before', 'after', 'source', 'op', 'ts_ms', 'transaction']

  op: r (c=create, u=update, d=delete, r=read)
  ts_ms: 1771860296473

  source.connector: postgresql
  source.db: webshop
  source.schema: public
  source.table: customers
  source.lsn: 27035888

  before: null

  after: {
    "customer_id": 1,
    "name": "Kovács Anna",
    "email": "kovacs.anna@example.com",
    "city": "Budapest",
    "segment": "premium",
    "created_at": 1771841676869,
    "updated_at": 17718416


In [7]:
# Takarítás
cur.execute(f"DELETE FROM customers WHERE email = 'demo.bela@example.com'")
print("Demo adatok törölve.")

cur.close()
conn.close()

print("\n=== Demo 1b összefoglalás ===")
print("1. Debezium connector regisztrálva → PostgreSQL WAL-ból olvassa a változásokat")
print("2. Minden INSERT/UPDATE/DELETE → Kafka topic-ba kerül")
print("3. Az események tartalmazzák a before/after állapotot")
print("4. A kafka-ui felületen (http://localhost:8080) vizuálisan is megnézhetők a topic-ok")
print("\nKövetkező: Demo 2 – Batch ETL/ELT pipeline")

Demo adatok törölve.

=== Demo 1b összefoglalás ===
1. Debezium connector regisztrálva → PostgreSQL WAL-ból olvassa a változásokat
2. Minden INSERT/UPDATE/DELETE → Kafka topic-ba kerül
3. Az események tartalmazzák a before/after állapotot
4. A kafka-ui felületen (http://localhost:8080) vizuálisan is megnézhetők a topic-ok

Következő: Demo 2 – Batch ETL/ELT pipeline
